# Pipeline 2: Video Caption Model Training

Train a **VideoCaptionModel** (BiGRU encoder + Bahdanau attention + LSTM decoder) on the **MSR-VTT** dataset.

**Dataset:** MSR-VTT — 10,000 videos with ~200,000 human captions (20 per video)

**Feature Extraction:** ResNet18 → 512-dim vectors per 2-sec segment

**Model:** Seq2Seq with Bahdanau Attention
- Encoder: BiGRU (2 layers, 512 hidden, bidirectional)
- Attention: Bahdanau (256-dim)
- Decoder: LSTM (1 layer, 512 hidden)
- Embedding: 256-dim

**Output:** `caption_model.pt` + `caption_vocab.pkl` saved to Google Drive

## Cell 1: Install Dependencies

In [ ]:
!pip install -q torch torchvision numpy opencv-python-headless Pillow nltk matplotlib

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

## Cell 2: Mount Google Drive & Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = '/content/drive/MyDrive/ai-video-summarizer'
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
MSVTT_DIR = os.path.join(DATA_DIR, 'raw', 'msvtt')
MSVTT_VIDEO_DIR = os.path.join(MSVTT_DIR, 'TrainValVideo')
MSVTT_ANNO_FILE = os.path.join(MSVTT_DIR, 'train_val_videodatainfo.json')
PROCESSED_DIR = os.path.join(DATA_DIR, 'processed', 'msvtt')
MODEL_DIR = os.path.join(PROJECT_DIR, 'outputs', 'models')

DATASET_PATH = os.path.join(PROCESSED_DIR, 'dataset.pkl')
VOCAB_PATH = os.path.join(MODEL_DIR, 'caption_vocab.pkl')
MODEL_PATH = os.path.join(MODEL_DIR, 'caption_model.pt')

for d in [PROCESSED_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Project dir: {PROJECT_DIR}')
print(f'Videos dir exists: {os.path.exists(MSVTT_VIDEO_DIR)}')
print(f'Annotations exist: {os.path.exists(MSVTT_ANNO_FILE)}')
print(f'Dataset exists: {os.path.exists(DATASET_PATH)}')

## Cell 3: Load MSR-VTT Annotations & Extract Features

**Important:** Upload MSR-VTT videos to `data/raw/msvtt/TrainValVideo/` and annotation JSON to `data/raw/msvtt/` in your Google Drive.

If you already have `dataset.pkl`, this cell will skip feature extraction.

In [ ]:
import json
import pickle
import numpy as np
import cv2
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torchvision.models as models
import torchvision.transforms as transforms

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEGMENT_SECONDS = 2
MAX_CAPTION_LEN = 30
MIN_WORD_FREQ = 2

print(f'Device: {DEVICE}')

if os.path.exists(DATASET_PATH):
    print(f'Dataset already exists at {DATASET_PATH} — skipping extraction')
    print('Loading existing dataset...')
    with open(DATASET_PATH, 'rb') as f:
        dataset = pickle.load(f)
    print(f'Loaded {len(dataset)} samples')
else:
    print('Building dataset from scratch...')
    
    # Load annotations
    with open(MSVTT_ANNO_FILE, 'r') as f:
        data = json.load(f)
    
    video_captions = {}
    for sentence in data['sentences']:
        vid = sentence['video_id']
        cap = sentence['caption'].strip()
        video_captions.setdefault(vid, []).append(cap)
    
    print(f'Total videos in annotations: {len(video_captions)}')
    
    # Load ResNet18
    resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    resnet.fc = torch.nn.Identity()
    resnet.eval()
    resnet.to(DEVICE)
    
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    # Build vocabulary first
    from collections import Counter
    
    SPECIAL_TOKENS = {'<pad>': 0, '<sos>': 1, '<eos>': 2, '<unk>': 3}
    
    class Vocabulary:
        def __init__(self):
            self.word2idx = dict(SPECIAL_TOKENS)
            self.idx2word = {v: k for k, v in self.word2idx.items()}
            self.freq = Counter()
    
        def build(self, captions, min_freq=2):
            for cap in captions:
                for word in cap.lower().split():
                    self.freq[word] += 1
            for word, count in self.freq.items():
                if count >= min_freq and word not in self.word2idx:
                    idx = len(self.word2idx)
                    self.word2idx[word] = idx
                    self.idx2word[idx] = word
    
        def encode(self, caption, max_len=30):
            tokens = ['<sos>'] + caption.lower().split()[:max_len - 2] + ['<eos>']
            ids = [self.word2idx.get(t, self.word2idx['<unk>']) for t in tokens]
            ids += [0] * (max_len - len(ids))
            return ids[:max_len]
    
        def decode(self, ids):
            words = []
            for i in ids:
                word = self.idx2word.get(i, '<unk>')
                if word == '<eos>': break
                if word not in ('<pad>', '<sos>', '<unk>'): words.append(word)
            return ' '.join(words)
    
        def __len__(self):
            return len(self.word2idx)
    
    all_captions = [c for caps in video_captions.values() for c in caps]
    vocab = Vocabulary()
    vocab.build(all_captions, min_freq=MIN_WORD_FREQ)
    print(f'Vocabulary size: {len(vocab)}')
    
    with open(VOCAB_PATH, 'wb') as f:
        pickle.dump(vocab, f)
    print(f'Vocab saved → {VOCAB_PATH}')
    
    # Extract features and build dataset
    dataset = []
    skipped = 0
    
    for video_id in tqdm(sorted(video_captions.keys()), desc='Processing videos'):
        video_path = os.path.join(MSVTT_VIDEO_DIR, f'{video_id}.mp4')
        if not os.path.exists(video_path):
            skipped += 1
            continue
        
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        if fps <= 0:
            cap.release()
            skipped += 1
            continue
        
        frame_interval = max(1, int(fps * SEGMENT_SECONDS))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        features = []
        seg_idx = 0
        
        while True:
            start = seg_idx * frame_interval
            if start >= total_frames: break
            cap.set(cv2.CAP_PROP_POS_FRAMES, start)
            ret, frame = cap.read()
            if not ret: break
            img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            tensor = transform(img).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                feat = resnet(tensor).squeeze().cpu().numpy()
            features.append(feat)
            seg_idx += 1
        
        cap.release()
        
        if not features:
            skipped += 1
            continue
        
        feat_array = np.array(features, dtype=np.float32)
        
        for caption in video_captions[video_id]:
            encoded = vocab.encode(caption, max_len=MAX_CAPTION_LEN)
            dataset.append({
                'video_id': video_id,
                'features': feat_array,
                'caption': encoded,
                'caption_text': caption,
            })
    
    print(f'\nTotal samples: {len(dataset)} | Videos skipped: {skipped}')
    
    with open(DATASET_PATH, 'wb') as f:
        pickle.dump(dataset, f)
    print(f'Dataset saved → {DATASET_PATH}')

## Cell 4: Build/Load Vocabulary

In [ ]:
from collections import Counter

SPECIAL_TOKENS = {'<pad>': 0, '<sos>': 1, '<eos>': 2, '<unk>': 3}

class Vocabulary:
    def __init__(self):
        self.word2idx = dict(SPECIAL_TOKENS)
        self.idx2word = {v: k for k, v in self.word2idx.items()}
        self.freq = Counter()

    def build(self, captions, min_freq=2):
        for cap in captions:
            for word in cap.lower().split():
                self.freq[word] += 1
        for word, count in self.freq.items():
            if count >= min_freq and word not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx] = word

    def encode(self, caption, max_len=30):
        tokens = ['<sos>'] + caption.lower().split()[:max_len - 2] + ['<eos>']
        ids = [self.word2idx.get(t, self.word2idx['<unk>']) for t in tokens]
        ids += [0] * (max_len - len(ids))
        return ids[:max_len]

    def decode(self, ids):
        words = []
        for i in ids:
            word = self.idx2word.get(i, '<unk>')
            if word == '<eos>': break
            if word not in ('<pad>', '<sos>', '<unk>'): words.append(word)
        return ' '.join(words)

    def __len__(self):
        return len(self.word2idx)


if os.path.exists(VOCAB_PATH):
    with open(VOCAB_PATH, 'rb') as f:
        vocab = pickle.load(f)
    print(f'Loaded vocab: {len(vocab)} words')
else:
    # Build from dataset
    all_caps = [s['caption_text'] for s in dataset]
    vocab = Vocabulary()
    vocab.build(all_caps, min_freq=MIN_WORD_FREQ)
    with open(VOCAB_PATH, 'wb') as f:
        pickle.dump(vocab, f)
    print(f'Built and saved vocab: {len(vocab)} words')

## Cell 5: Prepare DataLoader

In [ ]:
import random
from torch.utils.data import Dataset, DataLoader

RANDOM_SEED = 42
BATCH_SIZE = 128
MAX_SEQ_LEN = 60
VAL_SPLIT = 0.1

random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)


class CaptionDataset(Dataset):
    def __init__(self, samples, max_seq_len=60):
        self.samples = samples
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        features = torch.tensor(s['features'], dtype=torch.float32)
        if len(features) > self.max_seq_len:
            features = features[:self.max_seq_len]
        caption = torch.tensor(s['caption'], dtype=torch.long)
        return features, caption, len(features)


def collate_fn(batch):
    features, captions, lengths = zip(*batch)
    max_len = max(lengths)
    feat_dim = features[0].shape[-1]
    padded = torch.zeros(len(features), max_len, feat_dim)
    for i, (f, l) in enumerate(zip(features, lengths)):
        padded[i, :l] = f
    return padded, torch.stack(captions), torch.tensor(lengths, dtype=torch.long)


random.shuffle(dataset)
split = int(len(dataset) * (1 - VAL_SPLIT))
train_set = CaptionDataset(dataset[:split], MAX_SEQ_LEN)
val_set = CaptionDataset(dataset[split:], MAX_SEQ_LEN)

n_workers = 2 if DEVICE == 'cuda' else 0
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=n_workers, pin_memory=(DEVICE == 'cuda'))
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False,
                        collate_fn=collate_fn, num_workers=n_workers, pin_memory=(DEVICE == 'cuda'))

print(f'Train: {len(train_set)} samples | Val: {len(val_set)} samples')
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

## Cell 6: Define VideoCaptionModel

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class VideoEncoder(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=512, num_layers=2, dropout=0.3):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers, batch_first=True,
                          bidirectional=True, dropout=dropout if num_layers > 1 else 0.0)
        self.fc = nn.Linear(hidden_dim * 2, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, features):
        outputs, hidden = self.gru(features)
        outputs = self.dropout(outputs)
        hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)
        hidden = torch.tanh(self.fc(hidden))
        return outputs, hidden


class BahdanauAttention(nn.Module):
    def __init__(self, encoder_dim, decoder_dim, attention_dim):
        super().__init__()
        self.encoder_att = nn.Linear(encoder_dim, attention_dim)
        self.decoder_att = nn.Linear(decoder_dim, attention_dim)
        self.full_att = nn.Linear(attention_dim, 1)

    def forward(self, encoder_outputs, decoder_hidden):
        enc_att = self.encoder_att(encoder_outputs)
        dec_att = self.decoder_att(decoder_hidden).unsqueeze(1)
        scores = self.full_att(torch.tanh(enc_att + dec_att)).squeeze(2)
        weights = F.softmax(scores, dim=1)
        context = (encoder_outputs * weights.unsqueeze(2)).sum(dim=1)
        return context, weights


class CaptionDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, encoder_dim, hidden_dim, attention_dim, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.attention = BahdanauAttention(encoder_dim, hidden_dim, attention_dim)
        self.gru = nn.GRU(embed_dim + encoder_dim, hidden_dim, 1, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, token, hidden, encoder_outputs):
        embedded = self.dropout(self.embedding(token))
        context, weights = self.attention(encoder_outputs, hidden.squeeze(0))
        gru_input = torch.cat([embedded, context], dim=1).unsqueeze(1)
        output, hidden = self.gru(gru_input, hidden)
        prediction = self.fc_out(self.dropout(output.squeeze(1)))
        return prediction, hidden, weights


class VideoCaptionModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, encoder_hidden=512, decoder_hidden=512,
                 attention_dim=256, input_dim=512, encoder_layers=2, dropout=0.3):
        super().__init__()
        self.encoder = VideoEncoder(input_dim, encoder_hidden, encoder_layers, dropout)
        self.decoder = CaptionDecoder(vocab_size, embed_dim, encoder_hidden * 2,
                                      decoder_hidden, attention_dim, dropout)

    def forward(self, features, captions, teacher_forcing_ratio=0.5):
        batch_size = features.size(0)
        max_len = captions.size(1)
        vocab_size = self.decoder.fc_out.out_features
        encoder_outputs, hidden = self.encoder(features)
        hidden = hidden.unsqueeze(0)
        outputs = torch.zeros(batch_size, max_len, vocab_size, device=features.device)
        input_token = captions[:, 0]
        for t in range(1, max_len):
            pred, hidden, _ = self.decoder(input_token, hidden, encoder_outputs)
            outputs[:, t] = pred
            use_teacher = torch.rand(1).item() < teacher_forcing_ratio
            input_token = captions[:, t] if use_teacher else pred.argmax(1)
        return outputs


# Hyperparameters
EMBED_DIM = 256
ENCODER_HIDDEN = 512
DECODER_HIDDEN = 512
ATTENTION_DIM = 256
INPUT_DIM = 512
ENCODER_LAYERS = 2
DROPOUT = 0.3
EPOCHS = 30
LR = 1e-3
TEACHER_FORCING = 0.5
CLIP_GRAD = 1.0

model = VideoCaptionModel(
    vocab_size=len(vocab),
    embed_dim=EMBED_DIM,
    encoder_hidden=ENCODER_HIDDEN,
    decoder_hidden=DECODER_HIDDEN,
    attention_dim=ATTENTION_DIM,
    input_dim=INPUT_DIM,
    encoder_layers=ENCODER_LAYERS,
    dropout=DROPOUT,
).to(DEVICE)

print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Vocab size: {len(vocab)}')
print(f'Device: {DEVICE}')

## Cell 7: Training Loop

In [ ]:
import math
from nltk.translate.bleu_score import corpus_bleu

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
criterion = nn.CrossEntropyLoss(ignore_index=0)

best_val_loss = float('inf')
start_epoch = 1
history = []

# Resume from checkpoint if available
if os.path.exists(MODEL_PATH):
    print('Loading checkpoint to resume training...')
    ckpt = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    start_epoch = ckpt['epoch'] + 1
    if 'optimizer_state' in ckpt:
        optimizer.load_state_dict(ckpt['optimizer_state'])
    if 'scheduler_state' in ckpt:
        scheduler.load_state_dict(ckpt['scheduler_state'])
    if 'best_val_loss' in ckpt:
        best_val_loss = ckpt['best_val_loss']
    print(f'Resuming from epoch {start_epoch} (best val loss: {best_val_loss:.4f})')

print(f'\nTraining for {EPOCHS - start_epoch + 1} epochs...\n')

for epoch in range(start_epoch, EPOCHS + 1):
    # Train
    model.train()
    total_loss, total_tokens = 0, 0
    for features, captions, _ in train_loader:
        features, captions = features.to(DEVICE), captions.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(features, captions, teacher_forcing_ratio=TEACHER_FORCING)
        output_flat = outputs[:, 1:].reshape(-1, outputs.size(-1))
        target_flat = captions[:, 1:].reshape(-1)
        loss = criterion(output_flat, target_flat)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD)
        optimizer.step()
        non_pad = (target_flat != 0).sum().item()
        total_loss += loss.item() * non_pad
        total_tokens += non_pad
    train_loss = total_loss / max(total_tokens, 1)

    # Eval
    model.eval()
    total_loss, total_tokens = 0, 0
    references, hypotheses = [], []
    with torch.no_grad():
        for features, captions, _ in val_loader:
            features, captions = features.to(DEVICE), captions.to(DEVICE)
            outputs = model(features, captions, teacher_forcing_ratio=0.0)
            output_flat = outputs[:, 1:].reshape(-1, outputs.size(-1))
            target_flat = captions[:, 1:].reshape(-1)
            loss = criterion(output_flat, target_flat)
            non_pad = (target_flat != 0).sum().item()
            total_loss += loss.item() * non_pad
            total_tokens += non_pad
            # BLEU
            preds = outputs.argmax(dim=-1)
            for i in range(len(captions)):
                ref = vocab.decode(captions[i].cpu().tolist()).split()
                hyp = vocab.decode(preds[i].cpu().tolist()).split()
                references.append([ref])
                hypotheses.append(hyp)

    val_loss = total_loss / max(total_tokens, 1)
    bleu = corpus_bleu(references, hypotheses)
    scheduler.step(val_loss)

    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'bleu': bleu,
    })

    print(f'Epoch {epoch:3d}/{EPOCHS} | Train {train_loss:.4f} | Val {val_loss:.4f} | '
          f'PPL {math.exp(val_loss):.1f} | BLEU-4 {bleu:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'best_val_loss': best_val_loss,
            'vocab_size': len(vocab),
            'config': {
                'embed_dim': EMBED_DIM,
                'encoder_hidden': ENCODER_HIDDEN,
                'decoder_hidden': DECODER_HIDDEN,
                'attention_dim': ATTENTION_DIM,
                'input_dim': INPUT_DIM,
                'encoder_layers': ENCODER_LAYERS,
                'dropout': DROPOUT,
            },
        }, MODEL_PATH)
        print(f'           -> best model saved (val={val_loss:.4f})')

print(f'\nDone! Best val loss: {best_val_loss:.4f}')

## Cell 8: Plot Loss Curves & BLEU-4

In [ ]:
import matplotlib.pyplot as plt

if history:
    epochs_list = [h['epoch'] for h in history]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss curves
    ax1.plot(epochs_list, [h['train_loss'] for h in history], label='Train Loss', color='#7c3aed', linewidth=2)
    ax1.plot(epochs_list, [h['val_loss'] for h in history], label='Val Loss', color='#06b6d4', linewidth=2)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Cross-Entropy Loss')
    ax1.set_title('Training & Validation Loss')
    ax1.legend()
    ax1.grid(alpha=0.3)
    
    # BLEU-4 curve
    ax2.plot(epochs_list, [h['bleu'] for h in history], label='BLEU-4', color='#10b981', linewidth=2)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('BLEU-4 Score')
    ax2.set_title('BLEU-4 Score over Training')
    ax2.legend()
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(PROJECT_DIR, 'caption_training_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f'\nFinal metrics:')
    print(f'  Best val loss: {best_val_loss:.4f}')
    print(f'  Best BLEU-4:   {max(h["bleu"] for h in history):.4f}')
else:
    print('No training history to plot')

## Cell 9: Generate Sample Captions on Validation Set

In [ ]:
# Load best model
ckpt = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state'])
model.eval()

print('Sample captions from validation set:\n')
print('=' * 80)

val_samples = dataset[split:split + 10]  # First 10 validation samples

for i, sample in enumerate(val_samples):
    features = sample['features']
    ground_truth = sample['caption_text']
    
    # Pad/truncate features
    n = len(features)
    if n > MAX_SEQ_LEN:
        indices = np.linspace(0, n - 1, MAX_SEQ_LEN, dtype=int)
        features = features[indices]
    elif n < MAX_SEQ_LEN:
        pad = np.zeros((MAX_SEQ_LEN - n, 512), dtype=np.float32)
        features = np.concatenate([features, pad], axis=0)
    
    feat_tensor = torch.tensor(features, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    
    with torch.no_grad():
        enc_out, hidden = model.encoder(feat_tensor)
        hidden = hidden.unsqueeze(0)
        sos_idx = vocab.word2idx.get('<sos>', 1)
        eos_idx = vocab.word2idx.get('<eos>', 2)
        input_token = torch.tensor([sos_idx], device=DEVICE)
        decoded = []
        for _ in range(MAX_CAPTION_LEN):
            pred, hidden, _ = model.decoder(input_token, hidden, enc_out)
            next_id = pred.argmax(1).item()
            if next_id == eos_idx: break
            decoded.append(next_id)
            input_token = torch.tensor([next_id], device=DEVICE)
    
    generated = vocab.decode(decoded)
    
    print(f'Video: {sample["video_id"]}')
    print(f'  Ground truth: {ground_truth}')
    print(f'  Generated:    {generated}')
    print()

## Cell 10: Save Model & Vocab to Drive

In [ ]:
print('Models saved to Google Drive:')
print(f'  {MODEL_DIR}/')
for f in sorted(os.listdir(MODEL_DIR)):
    if 'caption' in f:
        size_mb = os.path.getsize(os.path.join(MODEL_DIR, f)) / (1024 * 1024)
        print(f'    {f} ({size_mb:.1f} MB)')

print(f'\nTo use locally, download these files to:')
print(f'  your_project/outputs/models/')
print(f'\nFiles needed for inference:')
print(f'  - caption_model.pt    (trained VideoCaptionModel)')
print(f'  - caption_vocab.pkl   (vocabulary for encoding/decoding)')